In [3]:
# import libraries

import pandas as pd
import json
import requests
from doctr.models import ocr_predictor
from doctr.io import DocumentFile

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# 1. OCR: Extract text using DocTR
def extract_text_doctr(image_path):
    model = ocr_predictor(det_arch='db_resnet50', reco_arch='crnn_vgg16_bn', pretrained=True)
    doc = DocumentFile.from_images([image_path])
    result = model(doc)
    
    # Export to structured text (maintaining basic order)
    full_text = ""
    for page in result.pages:
        for block in page.blocks:
            for line in block.lines:
                line_text = " ".join([word.value for word in line.words])
                full_text += line_text + "\n"
    return full_text

In [1]:
# 2. EXTRACTION: Send to local Ollama (BioMistral or Llama-3)
def extract_with_local_llm(raw_text):
    prompt = f"""
    Extract data from this medical bill OCR text. 
    Respond ONLY with valid JSON.
    Fields: patient_name, provider_name, date, total_due, billing_codes (list).
    
    TEXT:
    {raw_text}
    """
    
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "cniongolo/biomistral", 
            "prompt": prompt,
            "stream": False,
            "format": "json"
        }
    )
    return json.loads(response.json()['response'])

In [ ]:
# 3. RUN PIPELINE
image_path = "medical_bill.jpg"
print("Reading document...")
raw_text = extract_text_doctr(image_path)
print("Extracting fields...")
structured_data = extract_with_local_llm(raw_text)

In [ ]:
# Save to Pandas
df = pd.DataFrame([structured_data])
df.to_csv("local_medical_extracts.csv", index=False)
print("Done! Saved to CSV.")